# 01 — Embeddings Basics

**Lab 05 · RAG with LangChain**

---

In this notebook you will learn:

1. What a text embedding is and why it is useful
2. How to generate embeddings with `BAAI/bge-m3` via LangChain
3. How to measure semantic similarity with **cosine similarity**
4. How to visualise an embedding space with PCA

This notebook is **standalone** — it does not depend on any other lab module.
All you need is the `uv` environment for this lab (`uv sync --all-extras`).

> **Why `BAAI/bge-m3`?**  
> It is a state-of-the-art multilingual model (100+ languages) that runs
> entirely locally — no API key required. The RAG app in this lab uses the
> same model to index and query documents.

## 1 — What is a text embedding?

A **text embedding** is a fixed-size vector of floating-point numbers that
represents the *meaning* of a piece of text.

```
"The cat sat on the mat."  →  [0.012, -0.453, 0.871, ..., 0.034]  (1024 floats)
"A feline rested on the rug."  →  [0.015, -0.448, 0.865, ..., 0.041]
"The stock market crashed."    →  [-0.312, 0.201, -0.543, ..., 0.198]
```

The key property: **semantically similar texts produce geometrically close
vectors**. The first two sentences above are paraphrases — their vectors will
be very close. The third is unrelated — its vector will be far away.

This geometric structure is what makes embeddings useful for:
- **Semantic search** — find documents by meaning, not keywords
- **Clustering** — group similar texts automatically
- **RAG retrieval** — find the chunks most relevant to a user question

## 2 — Load the embedding model

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# ~570 MB download on first run — cached locally afterwards
MODEL_NAME = "BAAI/bge-m3"

embeddings = HuggingFaceEmbeddings(model_name=MODEL_NAME)
print(f"Model loaded: {MODEL_NAME}")

## 3 — Generate embeddings and inspect the vector space

In [ ]:
sentences = [
    # Group A — astronomy
    "The Milky Way galaxy contains over 200 billion stars.",
    "Astronomers discovered a new exoplanet orbiting a distant star.",
    "Black holes warp spacetime due to their extreme gravitational pull.",
    # Group B — cooking
    "Sauté the onions in olive oil until they turn golden.",
    "Preheat the oven to 180°C before baking the bread.",
    "A pinch of saffron gives paella its characteristic colour.",
    # Group C — machine learning
    "Gradient descent minimises the loss function iteratively.",
    "A transformer model uses self-attention to encode sequences.",
    "Overfitting occurs when a model memorises training data.",
]

vectors = embeddings.embed_documents(sentences)

print(f"Number of sentences : {len(vectors)}")
print(f"Embedding dimension : {len(vectors[0])}")
print(f"First 5 values of sentence 0: {[round(v, 4) for v in vectors[0][:5]]}")

## 4 — Cosine similarity

Two vectors can be compared by measuring the **angle** between them.
Cosine similarity equals the cosine of that angle:

$$\text{similarity}(A, B) = \frac{A \cdot B}{\|A\| \cdot \|B\|}$$

| Score | Interpretation |
|-------|---------------|
| `1.0` | Identical direction — same meaning |
| `0.8–1.0` | Very similar |
| `0.5–0.8` | Related |
| `< 0.5` | Unrelated or opposite |

Unlike Euclidean distance, cosine similarity is **not affected by vector
magnitude** — only direction matters. This makes it robust for text of
varying lengths.

In [ ]:
import numpy as np


def cosine_similarity(a: list[float], b: list[float]) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# Spot-check: within-group vs cross-group similarity
pairs = [
    ("Astronomy ↔ Astronomy", sentences[0], sentences[1]),
    ("Cooking   ↔ Cooking  ", sentences[3], sentences[4]),
    ("ML        ↔ ML       ", sentences[6], sentences[7]),
    ("Astronomy ↔ Cooking  ", sentences[0], sentences[3]),
    ("Astronomy ↔ ML       ", sentences[0], sentences[6]),
    ("Cooking   ↔ ML       ", sentences[3], sentences[6]),
]

print(f"{'Pair':<28}  {'Similarity':>10}")
print("-" * 42)
for label, s1, s2 in pairs:
    i, j = sentences.index(s1), sentences.index(s2)
    sim = cosine_similarity(vectors[i], vectors[j])
    print(f"{label}  {sim:>10.4f}")

## 5 — Similarity matrix (heatmap)

In [ ]:
import matplotlib.pyplot as plt

mat = np.array(vectors)  # shape: (9, 1024)

# Normalise rows so dot product equals cosine similarity
norms = np.linalg.norm(mat, axis=1, keepdims=True)
mat_norm = mat / norms
sim_matrix = mat_norm @ mat_norm.T  # (9, 9)

# Short labels for the axes
labels = [
    "Stars", "Exoplanet", "Black hole",   # astronomy
    "Onions", "Oven", "Saffron",           # cooking
    "Gradient", "Transformer", "Overfit",  # ML
]

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim_matrix, vmin=0.3, vmax=1.0, cmap="YlOrRd")
plt.colorbar(im, ax=ax, label="Cosine similarity")

ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)

# Annotate cells
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center", fontsize=8)

# Draw group boundaries
for pos in [2.5, 5.5]:
    ax.axhline(pos, color="white", linewidth=2)
    ax.axvline(pos, color="white", linewidth=2)

ax.set_title("Cosine similarity matrix — 3 topic groups")
plt.tight_layout()
plt.show()

## 6 — Visualising the embedding space with PCA

1024 dimensions are impossible to visualise directly. **Principal Component
Analysis (PCA)** projects the vectors onto the 2 directions of maximum
variance — a lossy but intuitive 2D snapshot of the space.

In [ ]:
from matplotlib.lines import Line2D
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
coords = pca.fit_transform(mat_norm)  # shape: (9, 2)

colors = ["#4C9BE8"] * 3 + ["#E86B4C"] * 3 + ["#50C878"] * 3
group_labels = ["Astronomy"] * 3 + ["Cooking"] * 3 + ["Machine Learning"] * 3

fig, ax = plt.subplots(figsize=(8, 6))

for i, (_label, color, short) in enumerate(
    zip(group_labels, colors, labels, strict=False)
):
    ax.scatter(coords[i, 0], coords[i, 1], color=color, s=120, zorder=3)
    ax.annotate(
        short,
        xy=(coords[i, 0], coords[i, 1]),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=9,
    )

legend_elements = [
    Line2D(
        [0], [0], marker="o", color="w",
        markerfacecolor="#4C9BE8", label="Astronomy", markersize=10,
    ),
    Line2D(
        [0], [0], marker="o", color="w",
        markerfacecolor="#E86B4C", label="Cooking", markersize=10,
    ),
    Line2D(
        [0], [0], marker="o", color="w",
        markerfacecolor="#50C878", label="Machine Learning", markersize=10,
    ),
]
ax.legend(handles=legend_elements, loc="best")

var = pca.explained_variance_ratio_
ax.set_xlabel(f"PC1 ({var[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({var[1]:.1%} variance)")
ax.set_title("BAAI/bge-m3 embeddings — PCA projection (2D)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7 — Semantic search in practice

This is exactly how RAG retrieval works: embed a query, then find the
document chunks whose vectors are closest (highest cosine similarity).

In [ ]:
def semantic_search(
    query: str, corpus: list[str], top_k: int = 3
) -> list[tuple[float, str]]:
    query_vec = np.array(embeddings.embed_query(query))
    query_vec /= np.linalg.norm(query_vec)
    scores = mat_norm @ query_vec  # dot product = cosine similarity (normalised vecs)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(float(scores[i]), corpus[i]) for i in top_indices]


queries = [
    "How do neural networks learn?",
    "What can I cook for dinner?",
    "Tell me about space exploration.",
]

for query in queries:
    print(f"\nQuery: \"{query}\"")
    for score, sentence in semantic_search(query, sentences, top_k=3):
        print(f"  {score:.4f}  {sentence}")

## 8 — Key takeaways and connection to the RAG pipeline

| Concept | Role in this notebook | Role in the RAG lab |
|---|---|---|
| `BAAI/bge-m3` | Embeds 9 example sentences | Embeds every document chunk at index time and every user query at retrieval time |
| Cosine similarity | Manual calculation on small vectors | Computed at scale by **ChromaDB** (`similarity_search`) |
| Semantic search | Implemented from scratch (numpy) | Handled by `vectorstore.similarity_search(query, k=10)` in `app/query.py` |
| PCA projection | 2D visualisation for intuition | Not used in production — 1024D vectors stay as-is |

**What comes next:**
- `02_chromadb_indexing.ipynb` — persist embeddings in ChromaDB, explore chunking strategies
- `03_rag_pipeline.ipynb` — full LCEL chain: query expansion → retrieval → reranking → generation